# Synthetic Waveform Generation: Lognormal Signal & Noise Pipeline


This notebook implements a pipeline to generate synthetic 1D time-series data (waveforms). It simulates physical signals characterized by a **Lognormal distribution**—common in particle physics and peak-sensing electronics—embedded in a stochastic background of **non-gaussian noise**.

### 1. Mathematical Framework
The core signal is modeled as a shifted Lognormal Probability Density Function (PDF):

$$f(t; \mu, \sigma, \tau) = \frac{1}{\sigma (t-\tau) \sqrt{2\pi}} \exp\left( -\frac{(\ln(t-\tau) - \mu)^2}{2\sigma^2} \right)$$

where:
* $\mu$ (mu) & $\sigma$ (sigma): Shape parameters of the distribution.
* $\tau$ (shift): The temporal offset or arrival time of the signal.

**Non-gaussian Noise:** Modeled as a superposition of $N$ asynchronous lognormal fluctuations with alternating phases to simulate baseline instabilities.

### 2. Data Structure & Output
The script generates multiple compressed NumPy archives (`.npz`). 

Each file contains `num_samples_per_file` waveforms along with their respective ground truth metadata:

| Key | Description | Data Type |
| :--- | :--- | :--- |
| `signal` | Pure signal traces (noise-free) | `float32` |
| `noise` | Correlated noise traces | `float32` |
| `peak` | Maximum amplitude of the pure signal | `float32` |
| `peak_position` | Temporal index (bin) of the maximum | `float32` |
| `integral` | Total area under the signal curve | `float32` |
| `mu`, `sigma`, `shift` | Original parameters used for generation | `float32` |

### 3. Requirements
* `numpy`: High-performance array manipulation.
* `scipy.stats`: Lognormal PDF and PPF (Percent Point Function) calculations.

---

In [ ]:
import numpy as np
from scipy.stats import lognorm

# Generate a lognormal dataset

In [ ]:
def gen_time(nsteps,tmin,tmax):
    time = np.linspace(tmin, tmax, nsteps)
    return time

def gen_signal(time, amplitude, mu, sigma, shift):
    scale = np.exp(mu)
    signal = amplitude * lognorm.pdf(time,s=sigma,loc=shift,scale=scale)
    return signal

def gen_correlated_noise(time, amplitude):    
    mu=6
    sigma=0.2
    min_shift=-2000
    max_shift=10000
    shift = np.random.uniform(min_shift,max_shift)
    noise = lognorm.pdf(time,loc=shift,s=sigma,scale=np.exp(mu))
    for i in range(1,100):
        shift = np.random.uniform(min_shift,max_shift)
        noise += np.power(-1,(i%2)) * lognorm.pdf(time,loc=shift,s=sigma,scale=np.exp(mu))
    return amplitude*noise

def gen_traces_array(n_samples, noise_amplitude, n_bins=10000, bin_width=1):
    
    time = gen_time(n_bins,0,n_bins*bin_width)
    
    arr_noise = []
    arr_signal = []
    arr_characteristics = []
    
    for i in range(0,n_samples):
        
        mu = np.random.uniform(6.65,6.75)
        sigma = np.random.uniform(0.35,0.45)
        mode = np.exp(mu - sigma*sigma)
        value_at_mode = lognorm.pdf(mode, s=sigma, scale=np.exp(mu))
        
        max_amplitude = np.log10(1./value_at_mode)
        r = np.random.uniform(0,max_amplitude)
        signal_amplitude = np.power(10,r)
        
        # Percent point function (inverse of cdf — percentiles)
        t99 = lognorm.ppf(0.99, s=sigma, scale=np.exp(mu))
        shift = np.random.uniform(150*bin_width, n_bins*bin_width - 150*bin_width - t99)

        noise = gen_correlated_noise(time, amplitude=noise_amplitude) 
        signal = gen_signal(time, amplitude=signal_amplitude, mu=mu, sigma=sigma, shift=shift)
        characteristics = [noise_amplitude, signal_amplitude, mu, sigma, shift, value_at_mode*signal_amplitude, mode, t99]

        arr_noise.append(noise)
        arr_signal.append(signal)
        arr_characteristics.append(characteristics)
        
    arr_noise = np.array(arr_noise)
    arr_signal = np.array(arr_signal)
    arr_characteristics = np.array(arr_characteristics)
        
    return time, arr_noise, arr_signal, arr_characteristics


In [ ]:
num_samples_per_file = 1000
relative_noise_amplitude = 1.0

for i in range(0,15):

    time, noise, signal, characteristics = gen_traces_array(num_samples_per_file, relative_noise_amplitude)

    true_integrals = characteristics[:,1]
    true_peaks = characteristics[:,5]
    true_peak_positions = characteristics[:,6] + characteristics[:,4]
    noise_ampl_arr = characteristics[:,0]
    sign_ampl_arr = characteristics[:,1]
    mu_arr = characteristics[:,2]
    sigma_arr = characteristics[:,3]
    shift_arr = characteristics[:,4]
    t99_arr = characteristics[:,7]

    np.savez_compressed(f"./synthetic_waveforms_{i:02d}", 
                        signal=np.float32(signal), 
                        noise=np.float32(noise), 
                        peak=np.float32(true_peaks), 
                        peak_position=np.float32(true_peak_positions), 
                        integral=np.float32(true_integrals), 
                        noise_amplitude=np.float32(noise_ampl_arr), 
                        signal_amplitude=np.float32(sign_ampl_arr),
                        mu=np.float32(mu_arr), 
                        sigma=np.float32(sigma_arr), 
                        shift=np.float32(shift_arr), 
                        t99=np.float32(t99_arr)
                        )

    del time, noise, signal, characteristics